## Long-Term Forecast (ml-server)

This section provides a quick start for long-horizon forecast checks through the async `POST /predict` API.

Notebook runtime assumptions:
- if notebook runs on host kernel: use `http://localhost:8030`
- if notebook runs in docker network: use `http://model-server:8030`
- model is registered in MLflow and mapped by `model_id`
- object reference points to the source archive used by the model configuration

In [4]:
import os
import time
import json
import requests

CANDIDATE_API_URLS = [
    os.getenv("ML_SERVER_PREDICT_URL", "").strip(),
    "http://localhost:8030/predict",
    "http://host.docker.internal:8030/predict",
    "http://model-server:8030/predict",
]
CANDIDATE_API_URLS = [url for url in CANDIDATE_API_URLS if url]

MODEL_ID = os.getenv("ML_SERVER_MODEL_ID", "root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt")
OBJECT_REFERENCE = os.getenv(
    "ML_SERVER_OBJECT_REFERENCE",
    "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
)
MODEL_SELECTION = {
    "version_alias": os.getenv("ML_SERVER_VERSION_ALIAS", "Production")
}
POLL_INTERVAL_SEC = float(os.getenv("ML_SERVER_POLL_INTERVAL", "1"))
MAX_ATTEMPTS = int(os.getenv("ML_SERVER_MAX_ATTEMPTS", "60"))


def parse_response(resp: requests.Response) -> dict:
    try:
        return resp.json()
    except Exception:
        return {"raw_text": resp.text}


def detect_api_url(candidates: list[str]) -> str:
    errors: list[str] = []
    for url in candidates:
        base = url.rsplit("/predict", 1)[0]
        health_url = f"{base}/ui/runtime-status"
        try:
            r = requests.get(health_url, timeout=5)
            if r.status_code == 200:
                print(f"Using API URL: {url}")
                return url
            errors.append(f"{url} -> HTTP {r.status_code}")
        except Exception as error:
            errors.append(f"{url} -> {type(error).__name__}: {error}")
    raise RuntimeError("No reachable API URL. Tried:\n" + "\n".join(errors))


API_URL = detect_api_url(CANDIDATE_API_URLS)

start_payload = {
    "object_reference": OBJECT_REFERENCE,
    "model_id": MODEL_ID,
    "model_selection": MODEL_SELECTION,
}

print("[1/2] Start async predict task")
print(json.dumps(start_payload, ensure_ascii=False, indent=2))

start_resp = requests.post(API_URL, json=start_payload, timeout=60)
start_data = parse_response(start_resp)

if start_resp.status_code != 202:
    raise RuntimeError(
        f"Unexpected start status={start_resp.status_code}. "
        f"Response={json.dumps(start_data, ensure_ascii=False)}"
    )

if "task_id" not in start_data:
    raise RuntimeError(f"task_id was not returned: {start_data}")

task_id = start_data["task_id"]
print(f"task_id: {task_id}")

print("[2/2] Poll result")
last_response = None
for attempt in range(1, MAX_ATTEMPTS + 1):
    poll_resp = requests.post(API_URL, json={"task_id": task_id}, timeout=60)
    poll_data = parse_response(poll_resp)
    last_response = poll_data

    http_status = poll_resp.status_code
    api_state = poll_data.get("state") if isinstance(poll_data, dict) else None

    if http_status == 202:
        print(f"attempt {attempt}/{MAX_ATTEMPTS}: processing (state={api_state})")
        time.sleep(POLL_INTERVAL_SEC)
        continue

    print(f"finished with http_status={http_status}, state={api_state}")
    print(json.dumps(poll_data, ensure_ascii=False, indent=2))

    if http_status == 503:
        base_url = API_URL.rsplit("/predict", 1)[0]
        try:
            models_resp = requests.get(f"{base_url}/ui/models", timeout=30)
            models_data = parse_response(models_resp)
            print("/ui/models snapshot:")
            print(json.dumps(models_data, ensure_ascii=False, indent=2))
        except Exception as error:
            print(f"Could not fetch /ui/models: {error}")
        print("Hint: 503 usually means model is not registered in MLflow Registry for the selected alias/version.")

    break
else:
    raise TimeoutError(f"Predict task did not finish after {MAX_ATTEMPTS} attempts")

print("Final response:")
print(json.dumps(last_response, ensure_ascii=False, indent=2))

Using API URL: http://localhost:8030/predict
[1/2] Start async predict task
{
  "object_reference": "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
  "model_id": "root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt",
  "model_selection": {
    "version_alias": "Production"
  }
}
task_id: 360046efe51b44d5b80d1b00f46b36c8
[2/2] Poll result
attempt 1/60: processing (state=processing)
finished with http_status=503, state=done
{
  "status": 503,
  "message": "MLFLOW is not available, so it is impossible to take values ​​at this time. Error: No cached MLflow bundle available for model_id=root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt selector=Production",
  "object_reference": "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
  "task_id": "360046efe51b44d5b80d1b00f46b36c8",
  "state": "done"
}
/ui/models snapshot:
{
  "status": 200,
  "models": [],
  "total": 0,
  "updated_at": "2026-05-04T18:32:04.445